# Revamped Implementation of the Backwards Pass of our `Conv_Layer`

To start, lets quickly review what we have inside the current instantiation and forward pass of the code, and note what items are needed for the backwards pass. 

In [ ]:
import cupy as cp
from cupy.lib.stride_tricks import as_strided
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        #input_shape has form (batch_size, height, width, channels)
        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = cp.zeros(self.num_filters, dtype = cp.float32) * 0.01
        self.weight_regularizer_l1 = 0
        self.weight_regularizer_l2 = 0
        self.bias_regularizer_l1 = 0
        self.bias_regularizer_l2 = 0

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = cp.sqrt(cp.float32(2.0 / n))
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = (cp.random.randn(
            filter_size[0],         #height
            filter_size[1],         #width
            input_depth,            #depth 
            num_filters             #number of filters
        ).astype(cp.float32)* std)

        self.weights = self.filter_weights

    def forward(self, inputs, training):
        #Extract Input dimensions

        fH, fW = self.filter_size
        sH, sW = self.strides
        S, H_in, W_in, D_in = inputs.shape
        
        #Creating padding depending on padding = same, or padding = valid
        if self.padding == "same":
            self.forward_pad_h = (self.filter_size[0] - 1) // 2
            self.forward_pad_w = (self.filter_size[1] - 1) // 2
        else:            
            self.forward_pad_h = 0
            self.forward_pad_w = 0

        #We need integer output dimensions, so cast equations to int
        H_out = int((H_in + 2 * self.forward_pad_h - self.filter_size[0]) / self.strides[0] + 1)
        W_out = int((W_in + 2 * self.forward_pad_w - self.filter_size[1]) / self.strides[1] + 1)
        
        #(0, 0) -> don't touch the number of samples in the batch
        #(P, P) -> pad top and bottom pixels by P pixels (axis 1)
        #(P, P) -> pad left and right pixels by P pixels (axis 2)
        #(0, 0) -> don't pad depth. 
        #contstant -> add constant_values for the padded values
        padded_inputs = cp.pad(array = inputs, 
                            pad_width = ((0, 0), (self.forward_pad_h, self.forward_pad_h),
                                          (self.forward_pad_w, self.forward_pad_w), (0, 0)),
                            mode = 'constant',
                            constant_values = 0).astype(cp.float32, copy = False)

        #Create an output tensor of size (batch_size, H_out, W_out, C_out)
        self.output = cp.zeros((S, H_out, W_out, self.num_filters), dtype = cp.float32)

        #create our sliding window
        self.patches = as_strided(
            padded_inputs,
            shape=(S, H_out, W_out, fH, fW, D_in),
            strides=(
                padded_inputs.strides[0],       # step between samples
                padded_inputs.strides[1] * sH,  # step down a row
                padded_inputs.strides[2] * sW,  # step across a column
                padded_inputs.strides[1],       # move down 1 row inside patch
                padded_inputs.strides[2],       # move right 1 col inside patch
                padded_inputs.strides[3],       # step across channels
            )
        )

        #Keep the samples, h_out, w_out, and the number of channels out. But, iterate over the patch(x, y) with channels c, and with the number of filters d
        self.output = cp.einsum('shwxyc,xycd->shwd', self.patches, self.filter_weights)
        self.output += self.biases.reshape((1, 1, 1, self.num_filters)) 

        self.inputs = inputs
        self.padded_inputs = padded_inputs
        return self.output
        #save the output tensor using self. for backpropogation

    def backward(self, dvalues):

        #extract dvalues dimensions
        S, H_out, W_out, C_out = dvalues.shape
        fH, fW, C_in, C_out = self.filter_weights.shape
        sH, sW = self.strides
        H_padded, W_padded = self.padded_inputs.shape[1:3]

        #dbiases has shape c_out as we intend to add dvalues to each filter. 
        self.dbiases = cp.sum(dvalues, axis = (0 , 1, 2)) 
        
        self.dweights = cp.einsum("shwxyc, shwd -> xycd", self.patches, dvalues)

        padded_dinputs = cp.zeros_like(self.padded_inputs, dtype = cp.float32)

        contributions = cp.einsum("shwd, xycd -> shwxyc", dvalues, self.filter_weights)

        contributions = contributions.astype(cp.float32)
        
        scatter_contributions_kernel(
            contributions.ravel(),
            S, H_out, W_out, fH, fW, C_in,
            H_padded, W_padded, sH, sW,
            padded_dinputs.ravel()
        )
        #truncate our borders 
        if self.padding == "same":
            P = (fH - 1) // 2
            self.dinputs = padded_dinputs[:, P:-P, P:-P, :] 
        else:
            self.dinputs = padded_dinputs 
        return self.dinputs


## Information Needed to Compute `backward`

To start, we'll keep our code to have 
```python
    S, H_in, W_in, C_in = inputs.shape
    # and
    S, H_out, W_out, C_out = dvalues.shape
```

The goal of the backwards pass is to compute the gradient vector $\frac{\partial L }{\partial x}$ or `dinputs` of shape $(S, H_{\text{in}}, W_{\text{in}}, D_{\text{in}})$ . The chain rule dictates:

$$
\frac{\partial L}{\partial x}=(\frac{\partial y}{\partial x})^T \frac{\partial L}{\partial y}
$$

While normally this results in the Doubly Block Toeplitz (DBT), which would consume a massive amount of VRAM, we can use the geometric equvialent of a tranposed convolution via the dialate and convolve method. 

### Coding the Stride Logic

If `sH > 1 or sW > 1` we'll have to stretch out `dvalues` (our upstream gradient) in place by inserting zeros between elements using the formula:

$$
\text{dilated stride} = (H_{out} - 1) \times sH + 1
$$

where:
* $H_{out}$ : Our forward output or upstream gradient height
* $sH$: The stride height 

This formula will be applied for the width case as well. We'll then have to create a zero tensor for this dilated tensor, and then populate our actual values using dvalues. To do this second part, we can levrage the double colon syntax in Python's slicing notation `[start:stop:step]`

* Writing [::sH] means we'll start an index 0, end at wherever our height dimension is, but skip ahead by `sH` steps each time. The same will apply for `sW`. An example is shown below

In [ ]:
import cupy as cp 

dvalues = cp.array([[[[1], [2], [3]], 
                     [[2], [3], [4]], 
                     [[3], [4], [5]]]], dtype=cp.float32)

print("Original array:")
print(dvalues[0, :, :, 0])

sH, sW = 2, 2       #our pretend stride values in the forward pass
S, H_out, W_out, C_out = dvalues.shape 
dilated_H = (H_out - 1) * sH + 1
dilated_W = (W_out - 1) * sW + 1
dvalues_dilated = cp.zeros((S, dilated_H, dilated_W, C_out), dtype = dvalues.dtype) 
dvalues_dilated[:, ::sH, ::sW, :] = dvalues 



print("\nDilated array:")
print(dvalues_dilated[0, :, :, 0])

Original array:
[[1. 2. 3.]
 [2. 3. 4.]
 [3. 4. 5.]]

Dilated array:
[[1. 0. 2. 0. 3.]
 [0. 0. 0. 0. 0.]
 [2. 0. 3. 0. 4.]
 [0. 0. 0. 0. 0.]
 [3. 0. 4. 0. 5.]]


Now that we have sucessfully dilated the array, we want to resolve the overlap that occurs in each overlapping gradient has on a single `dinputs` value. In the  notes in `Backpropagation_Conv_v2` while we could take the tranpose of the DBT block and the find the resulting `dinputs` values, we can instead flip the filter weights themselves and perform the computation on our `dvalues_dilated`. 

To do this, we can use the `[start:stop:step]` indexing syntax and "flip" the kernel by writing -1 in the step field. This works because in each axis, we're reversing the position of the axis. The updated code will now be shown 

```python
if sH > 1 or sW > 1: 
    dilated_H = (H_out - 1) * sH + 1 
    dilated_W = (W_out - 1) * sW + 1
    dvalues_dilated = cp.zeros(S, dilated_H, dilated_W, C_out, dtype=dvalues.dtype)
else:
    dvalues_dilated = dvalues
    dilated_H, dilated_W = H_out, W_out

flipped_weights = filter_weights[::-1, ::-1, :, :] #fH, fW, C_in, num filters
```

## Coding the Logic for `dweights` and `dbiases`

Starting with `dbiases`, if we wanted to observe how a single bias element $b_{c_{out}}$ impacts the overall loss $L$, we need to look at every output pixel value that it was added to. If we take the partial derivative of an individual output pixel wrt. its bias we'll result in a constant 1:

$$
\frac{\partial y_{s, h, w, c_out}}{\partial b_{c_{out}}} = 1
$$

In an informal sense, you could think of the original elementwise multiplication being set to zero (as that is our dweights which we don't care about yet) and since our bias had a variable value (like $f(x) = x$ and not $f(x)= x^2$) its resulting value is 1. That means, the resulting dbiases value is the sum of our dvalues across every signle pixel place that specific bias was shared. 

```python 

dbiases = cp.sum(dvalues, axis = (0, 1, 2)) 
```

### What about `dweights`   

Now we are answering a slightly different question, how do our weight kernels impact our loss. Each `dinputs` value can be influenced by one all the way to `fH * fW` weight values (in other words the size of the kernel). We can actually use our `self.patches` formula from before. The layout is $(S, H_{out}, W_{out}, fH, fW, D_{in})$. Our `dvalues` is the upward gradient with shape $(S, H_{out}, W_{out}, C_{out})$. 

#### But how does `self.patches` Work?

The formula indicated that for a fixed filter position $(i, j)$ and an input channel $c_{in}$, we have to loop through every sample $s$, every window height step $h$, and every window width step $w$, multiplying the pixel by the downstream gradient element, and then adding them up. 
 
This means we are performing a tensor contraction over the axes $(S, H_{out}, W_{out})$.

* **Batch** ($s$): index 0 in patches $(S)$, index 0 in dvalues $(S)$.
* **Output Row** ($h$): index 1 in patches $(H_{out})$, index 1 in dvalues $(H_{out})$.
* **Output Col** ($w$): index 2 in patches $(W_{out})$, index 2 in dvalues $(W_{out})$.

These three dimensions are summed over, and we're left with a new tensor $(fH, fW, D_{in}, C_{out})$ which are the exact dimensions of the weight tensor. 

```python 

dweights = cp.tensordot(self.patches, dvalues, axes=([0, 1, 2], [0, 1, 2]))
```

In the line above, because we only leave the axes that involve the filter dimensions, number of filters, and the channels out, we arrive at the final `dweights` needed for further use. It works as each indiivdual weight coefficient inside a filter is responsible for multiplying a specific, constant pixel offset within an input patch during the forward pass. In `self.patches`, we are telling the computer to look at each input pixel by the window step layout. 

### Coding the padding logic

Our goal is still to force our `dvalues` $H_{out}\times W_{out}$ values to transform into the sizes of $H_{in}\times W_{in}$. Performing some algebra using the original padding formula, along the formulas to find $H_{out}$, $H_{dilated}$, $H_{in}$, and $H_{padded}$ can result in us solving for backward padding $(P_b)$. After solving we arrive at two cases:

- Case A: `valid` Padding (Forward padding `P` = 0)
    We can use the formula below and plug in P = 0
        $$
    P_b = (fH - 1) - 0 = fH - 1
        $$

- Case B: `same` Padding (Forward padding `P` > 0)
    Using that same formula, we'd arrive at the result:
        $$
    P_b = (fH - 1) - P_f
        $$
    Because the forward pass already had padded the image keeping $h_{out}$ and $h_{in}$ the same, our backwrads pass now needs les padding such that the output **loses** size. 

Now that we have the exact formulas needed, we can create a simple if-else block to handle this logic. Finally, we'll use the built in `cp.pad` method to actually pad our height and width dimensions before performing the computations for `dinputs`. 

```python
if  self.padding == 'same': 

    backward_pad_h = (self.filter_size[0] - 1) - self.forward_pad_h
    backward_pad_w = (self.filter_size[1] - 1) - self.forward_pad_w
if self.padding == 'valid': 

    backward_pad_h = self.filter_size[0] - 1
    backward_pad_w = self.filter_size[1] - 1 

padded_dvalues = cp.pad(dvalues_dilated, ((0, 0), (backward_pad_h, backward_pad_h), (backward_pad_w, backward_pad_w), (0, 0)))
```

### Code for `dinputs`

We're now almost ready to sovle for `dinputs`. We already have our flipped weight kernel, `flipped_weights` and already have `dbiases`, the upstream gradient `dvalues`, and finally the upstream gradient set in its proper form based on the stride and padding `dvalues_padded`. We also have our `self.patches` which acts as a **read only memory view** into our tensor `dvalues_padded`. We won't use the same variable as we'll have to replace the array we're inputting into `dvalues_padded`, below is the written code for this section:

```python

dvalues_patches = cp.as_strided(
    dvalues_padded, 
    shape = (S, H_in, W_in, fH, fW, C_out),
    strides = (
        dvalues_padded.strides[0],
        dvalues_padded.strides[1],  # stride is always 1 in backward conv
        dvalues_padded.strides[2],  # stride is always 1 in backward conv
        dvalues_padded.strides[1],
        dvalues_padded.strides[2],
        dvalues_padded.strides[3]
    )
)
```

Finally, we can use these views into dvalues as the steps taken throughout the sliding window operation, and perform a tensordot operation where we'll take the summation of window using the window values and the weight values in `flipped_weights`; finally, we'll then add our `dbiases` result and we'll have arrived at `dinputs`. In this case, if `dvalues_padded` has shape $(S, H_{in}, W_{in}, fH, fW, C_{out})$ and `flipped_weights` has shape $(fH, fW, C_{in}, C_{out})$ then we have to line up our filter axes. For us, we need to sum over the input kernels so $(fH, fW, C_{out})$. Then, the summation result will be of shape $(S, H_{in}, W_{in}, C_{out})$ which is the correct shape. 
```python

dinputs = cp.tensordot(dvalues_patches, flipped_weights, axes = ([3, 4, 5], [0, 1, 2]))

```

We have written everything needed for the optimized backwards pass!

In [ ]:
import cupy as cp
from cupy.lib.stride_tricks import as_strided
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        #input_shape has form (batch_size, height, width, channels)
        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = cp.zeros(self.num_filters, dtype = cp.float32) * 0.01
        self.weight_regularizer_l1 = 0
        self.weight_regularizer_l2 = 0
        self.bias_regularizer_l1 = 0
        self.bias_regularizer_l2 = 0

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = cp.sqrt(cp.float32(2.0 / n))
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = (cp.random.randn(
            filter_size[0],         #height
            filter_size[1],         #width
            input_depth,            #depth 
            num_filters             #number of filters
        ).astype(cp.float32)* std)

        self.weights = self.filter_weights

    def forward(self, inputs, training):
        #Extract Input dimensions

        fH, fW = self.filter_size
        sH, sW = self.strides
        S, H_in, W_in, D_in = inputs.shape
        
        #Creating padding depending on padding = same, or padding = valid
        if self.padding == "same":
            self.forward_pad_h = (self.filter_size[0] - 1) // 2
            self.forward_pad_w = (self.filter_size[1] - 1) // 2
        else:            
            self.forward_pad_h = 0
            self.forward_pad_w = 0

        #We need integer output dimensions, so cast equations to int
        H_out = int((H_in + 2 * self.forward_pad_h - self.filter_size[0]) / self.strides[0] + 1)
        W_out = int((W_in + 2 * self.forward_pad_w - self.filter_size[1]) / self.strides[1] + 1)
        
        #(0, 0) -> don't touch the number of samples in the batch
        #(P, P) -> pad top and bottom pixels by P pixels (axis 1)
        #(P, P) -> pad left and right pixels by P pixels (axis 2)
        #(0, 0) -> don't pad depth. 
        #contstant -> add constant_values for the padded values
        padded_inputs = cp.pad(array = inputs, 
                            pad_width = ((0, 0), (self.forward_pad_h, self.forward_pad_h),
                                          (self.forward_pad_w, self.forward_pad_w), (0, 0)),
                            mode = 'constant',
                            constant_values = 0).astype(cp.float32, copy = False)

        #Create an output tensor of size (batch_size, H_out, W_out, C_out)
        self.output = cp.zeros((S, H_out, W_out, self.num_filters), dtype = cp.float32)

        #create our sliding window
        self.patches = as_strided(
            padded_inputs,
            shape=(S, H_out, W_out, fH, fW, D_in),
            strides=(
                padded_inputs.strides[0],       # step between samples
                padded_inputs.strides[1] * sH,  # step down a row
                padded_inputs.strides[2] * sW,  # step across a column
                padded_inputs.strides[1],       # move down 1 row inside patch
                padded_inputs.strides[2],       # move right 1 col inside patch
                padded_inputs.strides[3],       # step across channels
            )
        )

        #Keep the samples, h_out, w_out, and the number of channels out. But, iterate over the patch(x, y) with channels c, and with the number of filters d
        self.output = cp.einsum('shwxyc,xycd->shwd', self.patches, self.filter_weights)
        self.output += self.biases.reshape((1, 1, 1, self.num_filters)) 

        self.inputs = inputs
        self.padded_inputs = padded_inputs
        return self.output
        #save the output tensor using self. for backpropogation

    def backward(self, dvalues):

        #extract dvalues dimensions
        S, H_out, W_out, C_out = dvalues.shape
        fH, fW, C_in, C_out = self.filter_weights.shape
        sH, sW = self.strides
        _, H_in, W_in, _ = self.inputs.shape 

        #Now we need to account for dbiases and dweights

        self.dbiases = cp.sum(dvalues, axis = [0, 1, 2])

        self.dweights = cp.tensordot(self.patches, dvalues, axes = ([0, 1, 2], [0, 1, 2]))

        dilated_H = (H_out - 1) * sH + 1
        dilated_W = (W_out - 1) * sW + 1 
        
        dvalues_dilated = cp.zeros(shape= (S, dilated_H, dilated_W, C_out), dtype= dvalues.dtype)
        # Inject values using step slices
        dvalues_dilated[:, ::sH, ::sW, :] = dvalues
        
        # padding 
        if self.padding == "same": 
            backward_pad_h = (fH - 1) - self.forward_pad_h
            backward_pad_w = (fW - 1) - self.forward_pad_w
        if self.padding == "valid": 
            backward_pad_h = (fH - 1)
            backward_pad_w = (fW - 1)
        
        dvalues_padded = cp.pad(dvalues_dilated, pad_width= ((0, 0), (backward_pad_h, backward_pad_h), (backward_pad_w, backward_pad_w), (0, 0)))

        # flip the values in the fH and fW dimensions, leave C_in and C_out dimensions alone
        flipped_weights = self.filter_weights[::-1, ::-1, :, :]
        dvalues_patches = as_strided(dvalues_padded, 
                            shape = (S, H_in, W_in, fH, fW, C_out),
                            strides=(
                                dvalues_padded.strides[0],  # Batch step
                                dvalues_padded.strides[1],  # Window grid row step (backward stride = 1)
                                dvalues_padded.strides[2],  # Window grid col step (backward stride = 1)
                                dvalues_padded.strides[1],  # Internel window pixel row step
                                dvalues_padded.strides[2],  # Internel window pixel col step
                                dvalues_padded.strides[3]   # Output channel step
                            ))
    
        self.dinputs = cp.tensordot(
            dvalues_patches, 
            flipped_weights, 
            axes=([3, 4, 5], [0, 1, 3]) # Match fH, fW, C_out
        )

        return self.dinputs